# 08 - Evaluating RAG Quality with RAGAS

Building a RAG system is one thing — knowing whether it's actually **good** is another. This notebook introduces systematic evaluation.

## Why Evaluate?

Without evaluation, you're flying blind:
- Is the retriever finding the right documents?
- Is the LLM hallucinating beyond the provided context?
- Did changing the chunk size actually improve answers?

## The RAGAS Framework

RAGAS (Retrieval-Augmented Generation Assessment) measures four dimensions:

| Metric | What it measures | Bad score means |
|--------|-----------------|----------------|
| **Faithfulness** | Does the answer stick to the retrieved context? | LLM is hallucinating |
| **Answer Relevancy** | Is the answer relevant to the question? | LLM is off-topic |
| **Context Precision** | Are the retrieved docs relevant to the question? | Retriever finds wrong docs |
| **Context Recall** | Are all relevant docs in the knowledge base retrieved? | Retriever misses relevant docs |

In [ ]:
import os, sys, tempfile
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

import pandas as pd
import matplotlib.pyplot as plt

## Step 1: Build the RAG Pipeline

In [ ]:
from rag_engine.loaders import load_documents
from rag_engine.chunking.strategies import chunk_documents
from rag_engine.vectorstore.chroma_store import add_documents
from rag_engine.retrieval.retriever import RetrieverFactory
from rag_engine.llm.provider import LLMProvider
from rag_engine.chains.rag_chain import build_rag_chain

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))
chunks = chunk_documents(md_docs + csv_docs, chunk_size=512, chunk_overlap=50)

temp_dir = tempfile.mkdtemp()
vectorstore = add_documents(chunks, collection_name="nb08", persist_directory=temp_dir)
retriever = RetrieverFactory.create_retriever(vectorstore, strategy="mmr", top_k=5)
llm = LLMProvider.get_llm(temperature=0.0)

print(f"Pipeline ready: {len(chunks)} chunks indexed")

## Step 2: Create an Evaluation Dataset

We need question/ground-truth pairs. These should be answerable from our sample data.

In [ ]:
from rag_engine.evaluation.evaluator import EvalSample

eval_set = [
    EvalSample(
        question="What problem does RAG solve?",
        ground_truth="RAG solves LLM limitations including knowledge cutoff, hallucination, and lack of domain-specific knowledge by retrieving relevant documents at inference time.",
    ),
    EvalSample(
        question="What are the main chunking strategies?",
        ground_truth="The main chunking strategies are fixed-size splitting, recursive splitting (respecting paragraph and sentence boundaries), and semantic splitting (grouping by embedding similarity).",
    ),
    EvalSample(
        question="What is MMR retrieval?",
        ground_truth="MMR (Maximal Marginal Relevance) balances relevance with diversity to avoid returning near-duplicate chunks.",
    ),
    EvalSample(
        question="What is HyDE?",
        ground_truth="HyDE (Hypothetical Document Embeddings) generates a hypothetical answer first, embeds it, and uses that embedding for retrieval instead of the query embedding.",
    ),
    EvalSample(
        question="What are the four RAGAS evaluation metrics?",
        ground_truth="The four RAGAS metrics are faithfulness, answer relevancy, context precision, and context recall.",
    ),
    EvalSample(
        question="What is a cross-encoder?",
        ground_truth="A cross-encoder takes a pair of texts as input and produces a relevance score. More accurate than bi-encoders but slower, used for re-ranking.",
    ),
    EvalSample(
        question="What is the attention mechanism?",
        ground_truth="Attention is a mechanism that allows models to focus on relevant parts of the input when producing output, computing weighted sums of value vectors based on query-key compatibility.",
    ),
    EvalSample(
        question="What is a vector database?",
        ground_truth="A vector database is a specialized database optimized for storing, indexing, and querying high-dimensional vectors, using algorithms like HNSW for fast retrieval.",
    ),
]

print(f"Evaluation set: {len(eval_set)} questions")

## Step 3: Run the RAG Pipeline on the Eval Set

In [ ]:
from rag_engine.evaluation.evaluator import run_rag_on_eval_set

eval_set = run_rag_on_eval_set(eval_set, retriever, llm)

# Show a sample
sample = eval_set[0]
print(f"Question: {sample.question}")
print(f"\nGround truth: {sample.ground_truth}")
print(f"\nRAG answer: {sample.answer}")
print(f"\nRetrieved contexts: {len(sample.contexts)}")
print(f"First context: {sample.contexts[0][:200]}...")

## Step 4: Simple Evaluation (No Extra Dependencies)

In [ ]:
from rag_engine.evaluation.evaluator import simple_evaluate

results_df = simple_evaluate(eval_set)
print(results_df[["question", "has_answer", "answer_length", "num_contexts"]].to_string(index=False))

print(f"\n--- Summary ---")
print(f"Questions answered: {results_df['has_answer'].sum()}/{len(results_df)}")
print(f"Avg answer length: {results_df['answer_length'].mean():.0f} chars")
print(f"Avg contexts retrieved: {results_df['num_contexts'].mean():.1f}")

## Step 5: RAGAS Evaluation (Optional)

For full RAGAS evaluation, install the extra dependencies:

```bash
pip install -e ".[eval]"
```

Then uncomment and run:

In [ ]:
# from rag_engine.evaluation.evaluator import evaluate_with_ragas
#
# ragas_df = evaluate_with_ragas(eval_set)
# print(ragas_df[["question", "faithfulness", "answer_relevancy",
#                  "context_precision", "context_recall"]].to_string(index=False))
#
# # Aggregate scores
# metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
# print("\n--- Aggregate Scores ---")
# for m in metrics:
#     print(f"{m}: {ragas_df[m].mean():.3f}")

## Visualizing Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Answer length distribution
ax1.barh(range(len(results_df)), results_df["answer_length"], color="steelblue")
ax1.set_yticks(range(len(results_df)))
ax1.set_yticklabels([q[:30] + "..." for q in results_df["question"]], fontsize=8)
ax1.set_xlabel("Answer Length (chars)")
ax1.set_title("Answer Length per Question")

# Contexts retrieved
ax2.barh(range(len(results_df)), results_df["num_contexts"], color="coral")
ax2.set_yticks(range(len(results_df)))
ax2.set_yticklabels([q[:30] + "..." for q in results_df["question"]], fontsize=8)
ax2.set_xlabel("Number of Contexts")
ax2.set_title("Contexts Retrieved per Question")

plt.tight_layout()
plt.show()

## Interpreting Results

| Metric | Low score diagnosis | Fix |
|--------|-------------------|-----|
| **Faithfulness** | LLM adds info not in context | Stricter prompt ("use ONLY context"), lower temperature |
| **Answer Relevancy** | Answer doesn't address the question | Better prompt engineering, check retrieved context |
| **Context Precision** | Retrieved docs aren't relevant | Improve chunking, try different chunk sizes, use re-ranking |
| **Context Recall** | Missing relevant docs | More documents, smaller chunks, different embedding model |

**Key insight:** Evaluation tells you **where** to improve. Low faithfulness? Fix the prompt. Low context precision? Fix the retriever. Without metrics, you're guessing.

**Next:** [09 - Advanced Techniques](./09_advanced_techniques.ipynb) — HyDE, multi-query, and re-ranking.